Note: we have anonymised this repo so this notebook might not work very well. We will release the fully working version upon acceptance

In [ ]:
import json
import pandas as pd
import ast

from sklearn.model_selection import train_test_split

In [ ]:
dialect = 'yorkshire' 
real_data_file = f'data/data/human_annotated/{dialect}_dataset_with_rubric_shuffled.json'
with open(real_data_file, 'r', encoding='utf-8') as f:
    real = json.load(f)
len(real)

1007

## train and test data split

### get real data train and test

In [3]:
import pandas as pd
labels = [item['Label'] for item in real]

criteria_keys = ['c1', 'c2a', 'c2b', 'c3', 'c4', 'c5']

rubric_rows = [
    {k: item['Rubric'].get(k, None) for k in criteria_keys}
    for item in real
]
df_rubric = pd.DataFrame(rubric_rows)

print("=== Rubric Criteria Distribution ===")
for k in criteria_keys:
    print(f"\n{k}:")
    # Convert to string for consistent sorting, but show original values
    counts = df_rubric[k].value_counts(dropna=False)
    # Sort by value as string, but print original value and count
    for val, count in sorted(counts.items(), key=lambda x: str(x[0])):
        print(f"  {repr(val)}: {count}")

=== Rubric Criteria Distribution ===

c1:
  0: 37
  1: 978

c2a:
  0: 71
  1: 944

c2b:
  0: 184
  1: 831

c3:
  0: 71
  1: 944

c4:
  0: 237
  1: 778

c5:
  0: 61
  1: 954


### data split and save

In [ ]:
### 800 vs. 200 split
train_set, test_set = train_test_split(real, test_size=0.2, random_state=42, stratify=labels)
with open(f"data/processed/{dialect}_real.json", 'w', encoding='utf-8') as f:
    json.dump(train_set, f, ensure_ascii=False, indent=2)
with open(f"data/processed/{dialect}_test.json", 'w', encoding='utf-8') as f:
    json.dump(test_set, f, ensure_ascii=False, indent=2)

# Load train800 and test_set
with open(f"data/processed/{dialect}_real.json", 'r', encoding='utf-8') as f:
    train800 = json.load(f)
with open(f"data/processed/{dialect}_test.json", 'r', encoding='utf-8') as f:
    test_set = json.load(f)

len(train800), len(test_set)

In [ ]:
# Stratified split: first 500 for train, remaining 300 for test
train_labels = [item['Label'] for item in train800]
train500, remain300 = train_test_split(
    train800, train_size=500, random_state=42, stratify=train_labels
)

# Combine remain300 and original test_set for new test set
test500 = remain300 + test_set

# Save new splits
with open(f"data/data/finetuning_data/{dialect}_train_500.json", 'w', encoding='utf-8') as f:
    json.dump(train500, f, ensure_ascii=False, indent=2)
with open(f"data/data/finetuning_data/{dialect}_test_500.json", 'w', encoding='utf-8') as f:
    json.dump(test500, f, ensure_ascii=False, indent=2)

In [ ]:
# Stratified split: first 500 for train, remaining 300 for test
train_labels = [item['Label'] for item in train800]
train200, remain600 = train_test_split(
    train800, train_size=200, random_state=42, stratify=train_labels
)

# Combine remain300 and original test_set for new test set
test800 = remain600 + test_set

# Save new splits
with open(f"data/data/finetuning_data/{dialect}_train_200.json", 'w', encoding='utf-8') as f:
    json.dump(train200, f, ensure_ascii=False, indent=2)
with open(f"data/data/finetuning_data/{dialect}_test_800.json", 'w', encoding='utf-8') as f:
    json.dump(test800, f, ensure_ascii=False, indent=2)

## SFT

In [ ]:
## create training dataset
from sklearn.model_selection import train_test_split
import json
from llmclient_qwen import *
from evaluation_prompts import *

import torch
import random
import os
torch.cuda.empty_cache()
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, TrainerCallback, TrainerState, TrainerControl, BitsAndBytesConfig, set_seed
from huggingface_hub import login
from datasets import Dataset, load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import json
import numpy as np
from trl import SFTTrainer
import sys

In [ ]:
base_dir = ...

dialect_code = 'en-UK-York'
dialect = 'yorkshire'
locale = 'Yorkshire English'

date_id = '20260122'
type = 'real'

In [ ]:
import os, json
from pathlib import Path
import torch
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer

repo_root = ...
crit_list = ["c1", "c2a","c2b","c3","c4","c5"]

model_id = "Qwen/Qwen3-8B"      # set yours
generation_temperature = 0
lr = 2e-5                                  # set yours
max_seq_len = 1024                         # reduce if needed
per_device_bs = 4                          # increase only if memory allows
grad_accum = 16                             # effective batch = per_device_bs * grad_accum

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

for crit in crit_list:
    checkpoint_dir = repo_root / f"exp/{date_id}_{dialect}_real/models/outputs_{crit}"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    # fresh model per criterion, sharded/offloaded
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=True,
        # device_map="auto",
        device_map={"": 0},           # keep the whole model on GPU:0
        torch_dtype="auto",
        low_cpu_mem_usage=True,
    )
    model.gradient_checkpointing_enable()
    model.generation_config.temperature = generation_temperature
    model.config.use_cache = False

    # Load data
    train_path = repo_root / f"data/data/finetuning_data/{dialect}_{type}.json"
    train_set = json.load(open(train_path, encoding="utf-8"))
    train_list, eval_list = train_test_split(train_set, test_size=0.2, random_state=42)
    train_df = pd.DataFrame(train_list).sample(frac=1.0, random_state=42).reset_index(drop=True)
    eval_df = pd.DataFrame(eval_list).sample(frac=1.0, random_state=42).reset_index(drop=True)

    # Map to text only; let SFTTrainer tokenize/truncate
    def formatting_prompts_func(entry):
        messages = get_evaluator_prompt_single_criteria(entry=entry, num_exemplars=0, exemplar_dataset=[], locale=locale,
                                         criterion=crit, request_reasons=False)
        label = entry["Rubric"][crit]
        messages.append({"role": "assistant", "content": f'{{"{crit}": {label}}}'})
        chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        return {"text": chat_text}

    train_dataset = Dataset.from_pandas(train_df).map(
        formatting_prompts_func, remove_columns=["Prompt", "Output", "Label", "Rubric"]
    )
    eval_dataset = Dataset.from_pandas(eval_df).map(
        formatting_prompts_func, remove_columns=["Prompt", "Output", "Label", "Rubric"]
    )
    print(train_dataset[0]['text'])
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    args = TrainingArguments(
        eval_strategy="epoch",
        per_device_train_batch_size=per_device_bs,
        gradient_accumulation_steps=grad_accum,
        gradient_checkpointing=True,
        learning_rate=lr,
        fp16=not use_bf16,
        bf16=use_bf16,
        num_train_epochs=10,                  # adjust
        save_strategy="epoch",
        logging_strategy="epoch",
        output_dir=str(checkpoint_dir),
        optim="adamw_torch",
        lr_scheduler_type="linear",
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        # tokenizer=tokenizer,
        # packing=False,
        # max_seq_length=max_seq_len,
        # dataset_text_field="text",
    )

    print(f"=== Training {crit} on {len(train_dataset)} samples ===")
    trainer.train()
    trainer.save_model(str(checkpoint_dir / "checkpoint-final"))
    tokenizer.save_pretrained(str(checkpoint_dir / "checkpoint-final"))
    del model, trainer
    torch.cuda.empty_cache()


In [ ]:
### evaluate finetuned model on test dataset
import json
from sklearn.metrics import accuracy_score, f1_score

test_n = 'test'
print("=== FINETUNED EVALUATION ===")
finetuned_stats = {}
test_set = json.load(open(f"data/data/finetuning_data/{dialect}_{test_n}.json", 'r', encoding='utf-8'))

checkpoint = {
    'c1': "checkpoint-35",
    'c2a': "checkpoint-35",
    'c2b': "checkpoint-35",
    'c3': "checkpoint-35",
    'c4': "checkpoint-35",
    'c5': "checkpoint-35",
}
for crit in ["c1","c2a","c2b","c3", "c4", "c5"]:
    finetuned_results = []
    MODEL = f"exp/{date_id}_{dialect}_real/sft500500/models/outputs_{crit}/{checkpoint[crit]}"
    params = {"max_tokens": 1500, "temperature": 0}
    llm = LLMClient(params, MODEL)

    for entry in test_set:
        prompt = get_evaluator_prompt_single_criteria(entry=entry, num_exemplars=0, exemplar_dataset=[], locale=locale,
                                         criterion=crit, request_reasons=False)
        response = retrieve(prompt, llm, DEFAULT_RESPONSE={crit: -1})
        finetuned_results.append(response[0][crit])
        entry[f"finetuned_{crit}"] = response[0][crit]
        with open(f"exp/{date_id}_{dialect}_real/finetuned_{crit}_results_{test_n}.jsonl", "a") as f:
             f.write(json.dumps(entry) + "\n")

    labels = [p["Rubric"][crit] for p in test_set]
    valid_pairs = [(l, r) for l, r in zip(labels, finetuned_results) if r != -1]
    valid_labels = [p[0] for p in valid_pairs]
    valid_results = [p[1] for p in valid_pairs]
    acc = accuracy_score(valid_labels, valid_results)
    f1 = f1_score(valid_labels, valid_results)
    fail_rate = sum(i == -1 for i in finetuned_results) / len(finetuned_results)

    print(f"{crit} acc: {acc:.4f}, f1: {f1:.4f}")
    print(f"{crit} failure rate: {fail_rate:.4f}")

    finetuned_stats[crit] = {"acc": acc, "f1": f1, "fail": fail_rate}


## DPO

In [ ]:
## create training dataset
from sklearn.model_selection import train_test_split
import json
from notebooks.llmclient_qwen import *
from evaluation_prompts import *
import torch
import random
import os
torch.cuda.empty_cache()
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, TrainerCallback, TrainerState, TrainerControl, BitsAndBytesConfig, set_seed
from huggingface_hub import login
from datasets import Dataset, load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import json
import numpy as np
from trl import SFTTrainer, DPOTrainer
import sys

In [ ]:
!nvidia-smi

In [ ]:
base_dir = ...

dialect_code = 'en-UK-Wes'
dialect = 'cornish'
locale = 'West Country (Cornish) English'

date_id = '20260218'
type = 'dpo'

In [ ]:
import os, json
from pathlib import Path
import torch
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import DPOTrainer, DPOConfig

repo_root = ...

model_id = "Qwen/Qwen3-8B"      # set yours
generation_temperature = 0
lr = 2e-5                                  # set yours
max_seq_len = 1024                         # reduce if needed
per_device_bs = 4                          # increase only if memory allows
grad_accum = 16                             # effective batch = per_device_bs * grad_accum

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

checkpoint_dir = repo_root / f"exp/{date_id}_{dialect}_{type}/models/outputs"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    device_map={"": 0},           # keep the whole model on GPU:0
    torch_dtype="auto",
    low_cpu_mem_usage=True,
    )
model.gradient_checkpointing_enable()
model.generation_config.temperature = generation_temperature
model.config.use_cache = False

# Load data
train_path = repo_root / f"data/data/finetuning_data/{dialect}_real.json"
train_set = json.load(open(train_path, encoding="utf-8"))

# Build DPO pairs
dpo_pairs = []
for entry in train_set:
    label = entry["Label"]
    chosen = '{"Label": 1}'
    rejected = '{"Label": 0}'
    if label == 0:
        chosen, rejected = rejected, chosen
    dpo_pairs.append({
        "prompt": get_evaluator_prompt_all_criteria(
            entry=entry, num_exemplars=0, exemplar_dataset=[], request_breakdown=False, locale=locale, request_reasons=False),
        "chosen": chosen,
        "rejected": rejected
    })

train_list, eval_list = train_test_split(dpo_pairs, test_size=0.2, random_state=42)
train_df = pd.DataFrame(train_list).sample(frac=1.0, random_state=42).reset_index(drop=True)
eval_df = pd.DataFrame(eval_list).sample(frac=1.0, random_state=42).reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
args = DPOConfig(
    eval_strategy="epoch",
    per_device_train_batch_size=per_device_bs,
    gradient_accumulation_steps=grad_accum,
    gradient_checkpointing=True,
    learning_rate=lr,
    fp16=not use_bf16,
    bf16=use_bf16,
    num_train_epochs=10,                  # adjust
    save_strategy="epoch",
    logging_strategy="epoch",
    output_dir=str(checkpoint_dir),
    optim="adamw_torch",
    lr_scheduler_type="linear",
)

class SimpleDPOCollator:
    def __init__(self, tokenizer, max_length=1024):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        prompts = [item["prompt"] for item in batch]
        chosen = [item["chosen"] for item in batch]
        rejected = [item["rejected"] for item in batch]
        prompt_tokens = self.tokenizer(prompts, padding=True, truncation=True, max_length=self.max_length, return_tensors="pt")
        chosen_tokens = self.tokenizer(chosen, padding=True, truncation=True, max_length=self.max_length, return_tensors="pt")
        rejected_tokens = self.tokenizer(rejected, padding=True, truncation=True, max_length=self.max_length, return_tensors="pt")
        return {
            "prompt_input_ids": prompt_tokens["input_ids"],
            "prompt_attention_mask": prompt_tokens["attention_mask"],
            "chosen_input_ids": chosen_tokens["input_ids"],
            "chosen_attention_mask": chosen_tokens["attention_mask"],
            "rejected_input_ids": rejected_tokens["input_ids"],
            "rejected_attention_mask": rejected_tokens["attention_mask"],
        }

collator = SimpleDPOCollator(tokenizer, max_length=max_seq_len)
trainer = DPOTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print(f"=== DPO Training on {len(train_dataset)} pairs ===")
trainer.train()
trainer.save_model(str(checkpoint_dir / "checkpoint-final"))
tokenizer.save_pretrained(str(checkpoint_dir / "checkpoint-final"))
del model, trainer
torch.cuda.empty_cache()

In [ ]:
### evaluate finetuned model on test dataset
import json
from sklearn.metrics import accuracy_score, f1_score

print("=== FINETUNED EVALUATION ===")
finetuned_stats = {}
test_set = json.load(open(f"data/processed/{dialect}_test.json", 'r', encoding='utf-8'))

finetuned_results = []
MODEL = f"exp/{date_id}_{dialect}_{type}/models/outputs/checkpoint-99"
params = {"max_tokens": 1500, "temperature": 0}
llm = LLMClient(params, MODEL)

for entry in test_set:
    prompt = get_evaluator_prompt_all_criteria(entry, num_exemplars=0, exemplar_dataset=[], request_breakdown=False, locale=locale, request_reasons=False)
    response, failure = retrieve(prompt, llm, assert_fn=lambda x: "Label" in x, max_tries=3)
    finetuned_results.append(response)
    entry["NoBreakdown"] = {"response": response, "failed": failure}
    with open(f"exp/{date_id}_{dialect}_{type}/finetuned_results.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
